# Design

Following Andrew Ng's advice in *Machine Learning Yearning* — "build your first system quickly, then iterate" — we start with a quick-and-dirty baseline.

## Table of Contents
- [Problem Structure](#Problem-Structure)
- [A Bag of Solutions](#A-Bag-of-Solutions)
  - [Missing values](#Missing-values)
  - [Missing labels](#Missing-labels)
  - [Classification](#Classification)
- [Features](#Features)
  - [Feature engineering](#Feature-engineering)
  - [Feature selection](#Feature-selection)
  - [Preprocessing](#preprocessing)
- [Train/Validation Split](#Train/Validation-Split)
- [Baseline Model](#Baseline-Model)
- [Evaluation](#Evaluation)

## Problem Structure

Our task can be divided into three high-level steps:

1. Handle missing features.
2. Handle missing labels.
3. Train a classifier.

## A Bag of Solutions

### Missing values

**Discussion.** Given that discarding incomplete rows is not viable and the competition's guidance to treat missing values with care, we do not include `SimpleImputer` as a candidate despite its simplicity, since mean/median/mode imputation would discard exactly the informative missingness signal the competition warns against. Candidates instead include `IterativeImputer` (sklearn's linear/Bayesian ridge MICE implementation), `miceforest` (tree-based MICE), R's `mice` package, KNN imputation, and treating missingness as its own category for categorical and binary features.

Feature imputation is part of the modeling pipeline rather than a one-time preprocessing step: it must be fit on the training fold and applied to the corresponding validation/test fold within each split, to avoid leakage. This has practical implications for method choice, since `miceforest` and full MICE procedures are computationally heavy and may not be feasible to refit inside every cross-validation fold; this cost should be weighed against simpler or category-based approaches when finalizing the pipeline.

### Missing labels

**Discussion.** Given the substantial fraction of unlabeled training observations, simply discarding them would forgo potentially useful information. We therefore consider several approaches for exploiting unlabeled observations:

- **Social-graph label propagation.** Propagate labels through the referral graph using its topology, with directed versus undirected treatment of the graph as an open choice given its semantic directionality.
- **KNN-based label inference.** Infer labels from local neighborhoods in feature space, using feature similarity rather than the social graph.
- **GNN-based label inference.** Use graph neural networks such as GCN, GAT, or GraphSAGE to infer labels from both social-graph structure and node features.
- **PU learning.** Model the unlabeled set as a mixture of positive and negative examples rather than assigning individual labels through a neighborhood structure.
- **Self-training/pseudo-labeling.** Iteratively use confident predictions from a classifier trained on labeled data to assign labels to unlabeled observations.

These approaches exploit different sources of information—social-graph structure, feature-space neighborhoods, node features, or the distribution of labeled and unlabeled observations—providing a range of candidate solutions to the missing-label problem.


**Directed or undirected?** The directed-versus-undirected treatment of the referral graph involves a trade-off. Preserving direction retains potentially informative signal about the underlying relationships, while treating the graph as undirected is simpler to work with and may better represent relationships that are inherently mutual, such as friendships or personal connections. In the EDA, connected components were analyzed using an undirected representation of the graph.

**Ambiguity of ghost users.** A separate complication affects candidates that rely on the social graph: ghost users present in the graph but absent from both train and test have no associated feature row, and may represent structurally different roles. Topology-only approaches such as social-graph propagation may therefore treat paths through these heterogeneous nodes as evidence of a relationship between candidates, even when the intermediary is not itself a candidate. Feature-based graph approaches such as GNNs face a different issue: they require some feature representation for ghost nodes that were never described by the feature set. Possible placeholder representations can be constructed, but would introduce additional modeling assumptions. These issues are left open and should be addressed when the graph-based candidates are developed further.

### Classification

**Discussion.** Candidates for classification include:

- **Tree-based methods.** Particularly suitable for tabular data. Both bagging methods such as random forests and extremely randomized trees, and boosting methods such as XGBoost, LightGBM, and CatBoost are candidates.
- **Linear methods.** Simple baseline models such as logistic regression and linear SVM.
- **SVM with nonlinear kernels.** Extends SVM to capture nonlinear decision boundaries.
- **Neural networks.** A simple fully connected network as a deep-learning baseline.
- **Graph neural networks.** GCN, GAT, and GraphSAGE, which incorporate graph structure directly into the classification model. These overlap conceptually with graph-based label propagation but learn the use of graph structure jointly with the classifier.

## Features

### Feature engineering

**Discussion.** We will initially use the features as provided, without additional feature engineering. Graph-derived features are excluded because the graph will be used directly for label propagation, and incorporating graph information as features would introduce additional handling for users outside the graph. Domain-specific feature engineering is also not practical because the features are anonymized and their semantics are unknown. Generic interaction features (e.g., polynomial features) could be explored later, but there is currently no evidence that their additional complexity is warranted. Consistent with the quick-and-dirty approach, we defer feature engineering unless validation reveals poor generalization that motivates additional model complexity.

### Feature selection

**Discussion.** We retain the available features, with one exception: feature_014 is almost completely determined by feature_006, and feature_006 contains substantially more information due to its 11 distinct levels, so we retain feature_006 and remove feature_014. More generally, the features are anonymized, so we have no domain knowledge that would allow us to identify irrelevant or redundant features based on their meaning. Although the EDA reveals associations between some features and the target, marginal associations alone are not sufficient grounds for feature removal, since a feature with weak marginal association may still provide useful information in combination with others.

### Preprocessing

**Encoding.** The features are already represented in an encoded form: binary features as binary values, ordinal categorical features as ordered values, and continuous features as numerical values. No nominal categorical features are present, so no additional categorical encoding is required.

**Power Transformation.** Features `feature_010`, `feature_015`, and `feature_016` are right-skewed, making them candidates for a power transformation (e.g. log, Box-Cox, or Yeo-Johnson) to reduce skew before use in models sensitive to distribution shape.

**Scaling.** Whether numerical and non-binary ordinal features are scaled depends on the model ultimately used: models sensitive to feature magnitude, such as linear models, require scaling, while tree-based models do not. If scaling is needed, it comes after any applicable power transformation, and applies equally to numerical and ordinal features, since what determines scaling relevance is a feature's numerical magnitude, not whether it is continuous or ordinal. `feature_018` is already in [0,1] and would not need scaling regardless.

## Data

**Shuffling.** The originally labeled training users are shuffled before constructing the train/validation split, using a fixed random seed for reproducibility. This avoids artifacts or dependencies associated with the original ordering of the users.

**Train/Validation Split.** Our initial choice is to use stratified cross-validation over the originally labeled training users, with the number of folds considered in the range of 5–10. In each fold, the validation labels are held out as ground truth and are not available to the modeling pipeline. All pipeline components that learn from the data, including feature imputation, label propagation, and classifier training, are fitted separately within each fold using only the corresponding training data. Each validation fold therefore contains approximately 4–8% of the full training population, while the originally unlabeled users remain available to the training pipeline. The provided test set is kept completely separate and is used only for final prediction.

## Baseline Model

We start with a simple baseline pipeline consisting of:

- **Feature imputation.** Missing features are imputed using iterative imputation. We use sklearn's `IterativeImputer` with its default settings.
- **Label propagation.** Missing labels are inferred using topology-only label propagation on the undirected social graph. We use sklearn's `LabelPropagation` as the initial algorithmic choice.
- **Remaining missing labels.** After label propagation, some train users remain unlabeled because they are unreachable under the connected-component formulation. We consider two options for these remaining unlabeled train users, to be examined and toggled during implementation: dropping them from training, or self-training, in which a classifier is first trained on the labels available after propagation and then used to predict labels for the remaining unlabeled users, using the same classifier algorithm selected for classification.
- **Classification.** We use a tree-based classifier, well suited to tabular data. Our initial algorithmic choice is sklearn's `HistGradientBoostingClassifier`.

## Evaluation

**Evaluation Metric.** The challenge evaluates submissions using a cost-based metric that reflects the operational costs of cheating detection. Predictions are assigned to three decision regions: auto-pass, manual review, or auto-block. The evaluation function automatically searches for the decision thresholds that minimize the total cost. The costs are:

- False negative, where cheating passes through: **\$600**
- False positive in the auto-block region: **\$300**
- False positive in the manual review region: **\$150**
- True positive requiring manual review: **\$5**
- Correct auto-pass or auto-block: **\$0**

The leaderboard score is the negative of the minimum total cost, so lower total cost corresponds to a higher score.
